# 范数与距离（Norms and Distances）

对应课程：`phases/01-math-foundations/14-norms-and-distances`

> 你的距离函数定义了什么算"相似"。选错了，下游的一切都会跟着出错。

本 notebook 把 `distances.py` 里的核心函数拆开：每个函数一组中文注释，后面跟一小段可运行实验。完整打印型 demo 仍在 `distances.py`。

**贯穿全课的模式：** 范数量一个向量有多大；距离量两个点有多远。同一个数据集，换距离，最近邻就会换人。


## 0. 依赖

只用标准库。


In [1]:
import math


## 1. 范数：一个向量有多大

$$
\|x\|_1=\sum |x_i|,\quad
\|x\|_2=\sqrt{\sum x_i^2},\quad
\|x\|_p=\bigl(\sum |x_i|^p\bigr)^{1/p},\quad
\|x\|_\infty=\max |x_i|
$$

永远有 $\|x\|_\infty \le \|x\|_2 \le \|x\|_1$。L1 稀疏（LASSO），L2 平滑（权重衰减）。


In [2]:
def l1_norm(x):
    """曼哈顿长度：各分量绝对值之和。"""
    return sum(abs(xi) for xi in x)


def l2_norm(x):
    """欧几里得长度。"""
    return math.sqrt(sum(xi ** 2 for xi in x))


def lp_norm(x, p):
    """p-范数。p=inf 时退化为 max。"""
    if p == float("inf"):
        return max(abs(xi) for xi in x)
    return sum(abs(xi) ** p for xi in x) ** (1 / p)


def linf_norm(x):
    """无穷范数：最大分量。"""
    return max(abs(xi) for xi in x)


v = [3, 4]
print("| [3,4] |  L1=", l1_norm(v), " L2=", l2_norm(v), " Linf=", linf_norm(v))
print("Linf <= L2 <= L1?", linf_norm(v) <= l2_norm(v) <= l1_norm(v))


| [3,4] |  L1= 7  L2= 5.0  Linf= 4
Linf <= L2 <= L1? True


## 2. 距离：两个点有多远

距离就是差向量的范数：$d_p(a,b)=\|a-b\|_p$。


In [3]:
def l1_distance(a, b):
    return sum(abs(ai - bi) for ai, bi in zip(a, b))


def l2_distance(a, b):
    return math.sqrt(sum((ai - bi) ** 2 for ai, bi in zip(a, b)))


def linf_distance(a, b):
    return max(abs(ai - bi) for ai, bi in zip(a, b))


a, b = [0, 0], [3, 4]
print("L1=", l1_distance(a, b), " L2=", l2_distance(a, b), " Linf=", linf_distance(a, b))


L1= 7  L2= 5.0  Linf= 4


## 3. 余弦：只看方向，不看长短

$$
\cos\theta = \frac{a\cdot b}{\|a\|_2\|b\|_2},\quad
d_{\cos}=1-\cos\theta
$$

检索 embedding 常用余弦：同一方向、不同长度仍算很近。L2 会把长向量判成远。


In [4]:
def dot_product(a, b):
    return sum(ai * bi for ai, bi in zip(a, b))


def cosine_similarity(a, b):
    """夹角余弦。零向量约定为 0。"""
    na, nb = l2_norm(a), l2_norm(b)
    if na == 0 or nb == 0:
        return 0.0
    return dot_product(a, b) / (na * nb)


def cosine_distance(a, b):
    return 1.0 - cosine_similarity(a, b)


print("正交 [1,0] vs [0,1]  cos=", cosine_similarity([1, 0], [0, 1]))
print("同向不同长 [1,1] vs [10,10]  cos=", round(cosine_similarity([1, 1], [10, 10]), 6))
print("  L2 距离=", round(l2_distance([1, 1], [10, 10]), 4), "  (长度差被 L2 惩罚)")


正交 [1,0] vs [0,1]  cos= 0.0
同向不同长 [1,1] vs [10,10]  cos= 1.0
  L2 距离= 12.7279   (长度差被 L2 惩罚)


## 4. 马氏距离：先按协方差把空间拉圆

$$
d_M(x,y)=\sqrt{(x-y)^T\Sigma^{-1}(x-y)}
$$

相关的方向上「远一点」才算远。$\Sigma=I$ 时退回 L2。


In [5]:
def invert_matrix(matrix):
    """高斯-约当求逆。"""
    n = len(matrix)
    aug = [row[:] + [1.0 if i == j else 0.0 for j in range(n)] for i, row in enumerate(matrix)]
    for col in range(n):
        max_row = max(range(col, n), key=lambda r: abs(aug[r][col]))
        aug[col], aug[max_row] = aug[max_row], aug[col]
        pivot = aug[col][col]
        for j in range(2 * n):
            aug[col][j] /= pivot
        for row in range(n):
            if row != col:
                factor = aug[row][col]
                for j in range(2 * n):
                    aug[row][j] -= factor * aug[col][j]
    return [row[n:] for row in aug]


def mahalanobis_distance(x, y, cov_matrix):
    n = len(x)
    diff = [xi - yi for xi, yi in zip(x, y)]
    inv = invert_matrix(cov_matrix)
    temp = [sum(diff[j] * inv[j][i] for j in range(n)) for i in range(n)]
    return math.sqrt(max(0, sum(temp[i] * diff[i] for i in range(n))))


mean = [0.0, 0.0]
# 高度相关的椭圆：x 和 y 一起变
cov = [[1.0, 0.9], [0.9, 1.0]]
along = [1.0, 1.0]     # 沿着相关方向
across = [1.0, -1.0]   # 垂直于相关方向
print("沿相关轴:", round(mahalanobis_distance(along, mean, cov), 4))
print("横切相关:", round(mahalanobis_distance(across, mean, cov), 4), "  (同样的 L2，马氏更远)")
print("L2 两者:", round(l2_distance(along, mean), 4), round(l2_distance(across, mean), 4))


沿相关轴: 1.026
横切相关: 4.4721   (同样的 L2，马氏更远)
L2 两者: 1.4142 1.4142


## 5. 集合与字符串：Jaccard、编辑距离

Jaccard：$|A\cap B|/|A\cup B|$。编辑距离：插入/删除/替换的最少次数（Levenshtein）。


In [6]:
def jaccard_similarity(set_a, set_b):
    if not set_a and not set_b:
        return 1.0
    return len(set_a & set_b) / len(set_a | set_b)


def edit_distance(s1, s2):
    """Levenshtein DP。"""
    m, n = len(s1), len(s2)
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(m + 1):
        dp[i][0] = i
    for j in range(n + 1):
        dp[0][j] = j
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if s1[i - 1] == s2[j - 1]:
                dp[i][j] = dp[i - 1][j - 1]
            else:
                dp[i][j] = 1 + min(dp[i - 1][j], dp[i][j - 1], dp[i - 1][j - 1])
    return dp[m][n]


print("Jaccard {cat,dog} vs {dog,bird}:", jaccard_similarity({"cat", "dog"}, {"dog", "bird"}))
print("edit kitten -> sitting:", edit_distance("kitten", "sitting"))


Jaccard {cat,dog} vs {dog,bird}: 0.3333333333333333
edit kitten -> sitting: 3


## 6. 分布距离：KL vs Wasserstein-1

KL 是「用 $q$ 编码 $p$ 多付多少 nat」，支撑集不重叠时是 $\infty$。一维 Wasserstein 是两条 CDF 之间的面积，不重叠仍有梯度——生成模型更爱它。


In [7]:
def kl_divergence(p, q):
    total = 0.0
    for pi, qi in zip(p, q):
        if pi > 0:
            if qi <= 0:
                return float("inf")
            total += pi * math.log(pi / qi)
    return total


def wasserstein_1d(p, q):
    """一维离散：两条 CDF 的 L1。"""
    cdf_p = cdf_q = 0.0
    acc = 0.0
    for pi, qi in zip(p, q):
        cdf_p += pi
        cdf_q += qi
        acc += abs(cdf_p - cdf_q)
    return acc


p = [0.9, 0.1, 0.0]
q = [0.0, 0.1, 0.9]
print("KL(p||q) =", kl_divergence(p, q), "  (有 0 支撑 -> inf)")
print("W1(p,q)  =", wasserstein_1d(p, q), "  (仍然有限)")


KL(p||q) = inf   (有 0 支撑 -> inf)
W1(p,q)  = 1.8   (仍然有限)


## 7. 最近邻：距离一换，邻居就换


In [8]:
def find_nearest_neighbor(query, dataset, distance_fn):
    best_idx, best_dist = 0, float("inf")
    for i, point in enumerate(dataset):
        d = distance_fn(query, point)
        if d < best_dist:
            best_idx, best_dist = i, d
    return best_idx, best_dist


dataset = [[0, 0], [10, 0], [1, 8]]
query = [2, 2]
for name, fn in [("L2", l2_distance), ("L1", l1_distance), ("cos", cosine_distance)]:
    idx, d = find_nearest_neighbor(query, dataset, fn)
    print(f"{name}: 最近是点 {dataset[idx]}  dist={d:.4f}")


L2: 最近是点 [0, 0]  dist=2.8284
L1: 最近是点 [0, 0]  dist=4.0000
cos: 最近是点 [1, 8]  dist=0.2106


## 对照表

| 函数 | 什么时候用 |
|------|------------|
| L1 / L2 / Linf | 稀疏 / 默认几何 / 最坏分量 |
| 余弦 | embedding 检索，忽略长度 |
| 马氏 | 特征相关、尺度不同 |
| Jaccard / 编辑距离 | 集合、字符串 |
| KL / Wasserstein | 概率分布；不重叠时选 W |
| `find_nearest_neighbor` | 距离一换，邻居就换 |

```bash
python distances.py
```
